# AUS Uni Funding

Create one unique (deduplicated) table for each funding file/year in `rates/`.


In [1]:
from pathlib import Path

import re

import pandas as pd

from IPython.display import display, Markdown



# Locate the rates folder whether notebook is run from workspace root or docs/

candidate_dirs = [

    Path.cwd() / "rates",

    Path.cwd().parent / "rates",

    Path("rates"),

    Path("../rates"),

]



rates_dir = next((path.resolve() for path in candidate_dirs if path.exists()), None)

if rates_dir is None:

    raise FileNotFoundError("Could not find the 'rates' folder. Run this notebook from the project workspace or docs folder.")



files = sorted(

    [

        path

        for path in rates_dir.iterdir()

        if path.suffix.lower() in {".csv", ".xlsx", ".xls"}

    ],

    key=lambda p: p.name.lower(),

)



if not files:

    raise FileNotFoundError(f"No CSV/XLSX files found in: {rates_dir}")





def extract_table_from_raw_excel(raw: pd.DataFrame) -> pd.DataFrame:

    raw = raw.dropna(axis=0, how="all").dropna(axis=1, how="all")

    if raw.empty:

        return pd.DataFrame()



    header_candidates = []

    for row_index, row in raw.iterrows():

        values = [str(value).strip() for value in row if pd.notna(value) and str(value).strip()]

        if len(values) < 3:

            continue



        joined = " ".join(values).lower()

        score = len(values)

        if any(keyword in joined for keyword in ["foe", "funding", "contribution", "student", "cluster"]):

            score += 10

        header_candidates.append((score, row_index))



    if not header_candidates:

        return pd.DataFrame()



    _, header_row = max(header_candidates, key=lambda item: item[0])

    header = raw.loc[header_row].astype(str).str.strip()

    table = raw.loc[header_row + 1 :].copy()

    table.columns = header



    valid_columns = [

        column

        for column in table.columns

        if column and column != "nan" and not str(column).lower().startswith("unnamed")

    ]

    table = table.loc[:, valid_columns]

    table = table.dropna(axis=0, how="all")



    return table.reset_index(drop=True)





def load_table(file_path: Path) -> pd.DataFrame:

    if file_path.suffix.lower() == ".csv":

        return pd.read_csv(file_path)



    excel_file = pd.ExcelFile(file_path)

    best_table = pd.DataFrame()

    best_score = -1



    for sheet_name in excel_file.sheet_names:

        raw_sheet = pd.read_excel(file_path, sheet_name=sheet_name, header=None)

        extracted = extract_table_from_raw_excel(raw_sheet)

        score = extracted.shape[0] * extracted.shape[1]



        if score > best_score:

            best_score = score

            best_table = extracted



    if best_score > 0:

        return best_table



    return pd.read_excel(file_path)





def make_table_name(file_path: Path) -> str:

    year_match = re.search(r"(20\d{2})", file_path.stem)

    year = year_match.group(1) if year_match else "unknown"

    stem_slug = re.sub(r"[^0-9a-zA-Z]+", "_", file_path.stem).strip("_").lower()

    return f"table_{year}_{stem_slug}"





def find_foe_column(df: pd.DataFrame):

    foe_candidates = [column for column in df.columns if "foe" in str(column).lower()]

    return foe_candidates[0] if foe_candidates else None





def find_description_column(df: pd.DataFrame, foe_column: str):

    description_keywords = ["description", "subject", "discipline", "field", "title", "name", "area"]

    excluded_keywords = ["funding", "cluster", "contribution", "student", "maximum", "commonwealth"]

    candidates = [column for column in df.columns if column != foe_column]



    keyword_match = [

        column

        for column in candidates

        if any(keyword in str(column).lower() for keyword in description_keywords)

        and not any(keyword in str(column).lower() for keyword in excluded_keywords)

    ]

    if keyword_match:

        return keyword_match[0]



    text_candidates = []

    for column in candidates:

        values = df[column].dropna().astype(str).str.strip()

        if values.empty:

            continue



        non_numeric_ratio = (~values.str.match(r"^[0-9.\-]+$", na=False)).mean()

        avg_length = values.str.len().mean()

        if non_numeric_ratio >= 0.7 and avg_length >= 6:

            text_candidates.append((avg_length, column))



    if text_candidates:

        text_candidates.sort(reverse=True)

        return text_candidates[0][1]



    return None





def normalize_foe(series: pd.Series) -> pd.Series:

    cleaned = (

        series.astype(str)

        .str.strip()

        .str.replace(r"\.0$", "", regex=True)

        .str.replace(r"[^0-9A-Za-z]", "", regex=True)

    )

    return cleaned.str.lstrip("0").replace("", "0")





unique_tables = {}



for file_path in files:

    data = load_table(file_path)

    unique_data = data.drop_duplicates().reset_index(drop=True)

    table_name = make_table_name(file_path)



    unique_tables[table_name] = unique_data

    globals()[table_name] = unique_data



print(f"Loaded {len(unique_tables)} unique tables from: {rates_dir}")

print("Tables created as variables:\n- " + "\n- ".join(unique_tables.keys()))



foe_description_lookup = {}



for table_name, table in unique_tables.items():

    year_match = re.search(r"table_(20\d{2})_", table_name)

    if not year_match:

        continue



    year = int(year_match.group(1))

    if year < 2021:

        continue



    foe_column = find_foe_column(table)

    if not foe_column:

        continue



    description_column = find_description_column(table, foe_column)

    if not description_column:

        continue



    lookup_rows = table[[foe_column, description_column]].dropna().copy()

    lookup_rows[foe_column] = normalize_foe(lookup_rows[foe_column])

    lookup_rows[description_column] = lookup_rows[description_column].astype(str).str.strip()

    lookup_rows = lookup_rows[(lookup_rows[foe_column] != "") & (lookup_rows[description_column] != "")]



    for foe_key, description in lookup_rows[[foe_column, description_column]].itertuples(index=False):

        if foe_key not in foe_description_lookup:

            foe_description_lookup[foe_key] = description





def build_clone_with_descriptions(source_table_name: str, clone_table_name: str):

    if source_table_name not in unique_tables:

        return None



    source_table = unique_tables[source_table_name].copy()

    foe_column = find_foe_column(source_table)

    if not foe_column:

        return None



    foe_key_series = normalize_foe(source_table[foe_column])

    source_table["FOE_Description"] = foe_key_series.map(foe_description_lookup)



    unique_tables[clone_table_name] = source_table

    globals()[clone_table_name] = source_table

    return source_table





table_2019_original = next((name for name in unique_tables if name.startswith("table_2019_")), None)

table_2020_original = next((name for name in unique_tables if name.startswith("table_2020_")), None)



table_2019_with_descriptions = build_clone_with_descriptions(table_2019_original, "table_2019_with_descriptions")

table_2020_with_descriptions = build_clone_with_descriptions(table_2020_original, "table_2020_with_descriptions")



for table_name, table in unique_tables.items():

    display(Markdown(f"### {table_name} ({len(table)} rows)"))

    display(table)


Loaded 11 unique tables from: C:\Users\neddp\ECC3479-Project-JRGS\reproduction test\rates
Tables created as variables:
- table_2019_2019_allocation_of_units_of_study
- table_2020_2020_allocation_of_units_of_study
- table_2021_2021_allocation_of_units_of_study_update07122021
- table_2022_2022_allocation_of_units_of_study_update07122021
- table_2023_2023_allocation_of_units_of_study
- table_2024_2024_allocation_of_units_of_study
- table_2025_2025_allocation_of_units_of_study
- table_2026_2026_allocation_of_units_of_study
- table_unknown_allocation_of_units_of_study_year_comparison
- table_unknown_allocation_of_units_of_study_year_comparison_by_foe_code
- table_unknown_combined_allocation_of_units_of_study


### table_2019_2019_allocation_of_units_of_study (418 rows)

,FundingCluster,E312of27,FOE,2019MaximumStudentContibution,2019CommonwealthContribution
0,Funding Cluster 3,NaN,90701,6566,10630
1,Funding Cluster 5,27.0,90701,6566,13073
2,Funding Cluster 1,NaN,80100,10958,2160
3,Funding Cluster 1,NaN,80101,10958,2160
4,Funding Cluster 1,NaN,80300,10958,2160
...,...,...,...,...,...
413,Funding Cluster 8,NaN,60799,10958,23590
414,Funding Cluster 8,NaN,61100,10958,23590
415,Funding Cluster 8,NaN,61101,10958,23590
416,Funding Cluster 8,NaN,61103,10958,23590


### table_2020_2020_allocation_of_units_of_study (418 rows)

,FundingCluster,E312of27,FOE,2020MaximumStudentContibution,2020CommonwealthContribution
0,Funding Cluster 3,NaN,90701,6684,10821
1,Funding Cluster 5,27.0,90701,6684,13308
2,Funding Cluster 1,NaN,80100,11155,2198
3,Funding Cluster 1,NaN,80101,11155,2198
4,Funding Cluster 1,NaN,80300,11155,2198
...,...,...,...,...,...
413,Funding Cluster 8,NaN,60799,11155,24014
414,Funding Cluster 8,NaN,61100,11155,24014
415,Funding Cluster 8,NaN,61101,11155,24014
416,Funding Cluster 8,NaN,61103,11155,24014


...

,year,foe_code,funding_cluster,special_course_type_code,maximum_student_contribution_indicator,funding_cluster_varies,maximum_student_contribution,commonwealth_contribution,grandfathered_maximum_student_contribution,grandfathered_commonwealth_contribution,detailed_discipline_title,detailed_discipline,narrow_discipline,broad_discipline
0,2019,10100.0,Funding Cluster 3,Not E312=27,Not E392=8,No,9359,10630,NaN,NaN,Mathematical Sciences,010100 - Mathematical Sciences,0101 - Mathematical Sciences,01 - NATURAL AND PHYSICAL SCIENCES
1,2019,10101.0,Funding Cluster 3,Not E312=27,Not E392=8,No,9359,10630,NaN,NaN,Mathematics,010101 - Mathematics,0101 - Mathematical Sciences,01 - NATURAL AND PHYSICAL SCIENCES
2,2019,10103.0,Funding Cluster 3,Not E312=27,Not E392=8,No,9359,10630,NaN,NaN,Statistics,010103 - Statistics,0101 - Mathematical Sciences,01 - NATURAL AND PHYSICAL SCIENCES
3,2019,10199.0,Funding Cluster 3,Not E312=27,Not E392=8,No,9359,10630,NaN,NaN,"Mathematical Sciences, n.e.c.","010199 - Mathematical Sciences, n.e.c.",0101 - Mathematical Sciences,01 - NATURAL AND PHYSICAL SCIENCES
4,2019,10300.0,Funding Cluster 7,Not E312=27,Not E392=8,No,9359,18586,NaN,NaN,Physics and Astronomy,010300 - Physics and Astronomy,0103 - Physics and Astronomy,01 - NATURAL AND PHYSICAL SCIENCES
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3479,2026,120505.0,Funding Cluster 1,Any E312 value,Any E392 value,No,17399,1316,13624.0,2682.0,Work Practices Programs,120505 - Work Practices Programs,1205-Employment Skills Programs,12 - MIXED FIELD PROGRAMMES
3480,2026,120599.0,Funding Cluster 1,Any E312 value,Any E392 value,No,17399,1316,13624.0,2682.0,"Employment Skills Programs, n.e.c.","120599 - Employment Skills Programs, n.e.c.",1205-Employment Skills Programs,12 - MIXED FIELD PROGRAMMES
3481,2026,129900.0,Funding Cluster 1,Any E312 value,Any E392 value,No,17399,1316,13624.0,2682.0,Other Mixed Field Programmes,129900 - Other Mixed Field Programmes,1299 - Other Mixed Field Programmes,12 - MIXED FIELD PROGRAMMES
3482,2026,129999.0,Funding Cluster 1,Any E312 value,Any E392 value,No,17399,1316,13624.0,2682.0,"Mixed Field Programmes, n.e.c.","129999 - Mixed Field Programmes, n.e.c.",1299 - Other Mixed Field Programmes,12 - MIXED FIELD PROGRAMMES


### table_2019_with_descriptions (418 rows)

,FundingCluster,E312of27,FOE,2019MaximumStudentContibution,2019CommonwealthContribution,FOE_Description
0,Funding Cluster 3,NaN,90701,6566,10630,Psychology (Not professional pathway psycholog...
1,Funding Cluster 5,27.0,90701,6566,13073,Psychology (Not professional pathway psycholog...
2,Funding Cluster 1,NaN,80100,10958,2160,Accounting
3,Funding Cluster 1,NaN,80101,10958,2160,Accounting
4,Funding Cluster 1,NaN,80300,10958,2160,Business and Management
...,...,...,...,...,...,...
413,Funding Cluster 8,NaN,60799,10958,23590,"Dental Studies, n.e.c."
414,Funding Cluster 8,NaN,61100,10958,23590,Veterinary Studies
415,Funding Cluster 8,NaN,61101,10958,23590,Veterinary Science
416,Funding Cluster 8,NaN,61103,10958,23590,Veterinary Assisting


### table_2020_with_descriptions (418 rows)

,FundingCluster,E312of27,FOE,2020MaximumStudentContibution,2020CommonwealthContribution,FOE_Description
0,Funding Cluster 3,NaN,90701,6684,10821,Psychology (Not professional pathway psycholog...
1,Funding Cluster 5,27.0,90701,6684,13308,Psychology (Not professional pathway psycholog...
2,Funding Cluster 1,NaN,80100,11155,2198,Accounting
3,Funding Cluster 1,NaN,80101,11155,2198,Accounting
4,Funding Cluster 1,NaN,80300,11155,2198,Business and Management
...,...,...,...,...,...,...
413,Funding Cluster 8,NaN,60799,11155,24014,"Dental Studies, n.e.c."
414,Funding Cluster 8,NaN,61100,11155,24014,Veterinary Studies
415,Funding Cluster 8,NaN,61101,11155,24014,Veterinary Science
416,Funding Cluster 8,NaN,61103,11155,24014,Veterinary Assisting


In [2]:
# Export combined annual funding table (2019-2026)

from pathlib import Path

import re





def pick_col(columns, include_keywords, exclude_keywords=None):

    exclude_keywords = exclude_keywords or []

    for col in columns:

        col_text = str(col).lower().replace("\n", " ")

        if all(keyword in col_text for keyword in include_keywords) and not any(keyword in col_text for keyword in exclude_keywords):

            return col

    return None





def get_table_by_year(year):

    with_desc_name = f"table_{year}_with_descriptions"

    if with_desc_name in unique_tables:

        return unique_tables[with_desc_name].copy()



    year_tables = [name for name in unique_tables if re.match(fr"table_{year}_", name)]

    if year_tables:

        return unique_tables[year_tables[0]].copy()

    return None





combined_rows = []



for year in range(2019, 2027):

    table = get_table_by_year(year)

    if table is None or table.empty:

        continue



    columns = list(table.columns)



    foe_col = pick_col(columns, ["foe"])

    desc_col = pick_col(columns, ["foe", "title"]) or pick_col(columns, ["description"]) or ("FOE_Description" if "FOE_Description" in table.columns else None)

    cluster_col = pick_col(columns, ["funding", "cluster"])

    max_student_col = pick_col(columns, ["maximum", "student"], ["grandfathered"])

    commonwealth_col = pick_col(columns, ["commonwealth", "contribution"], ["grandfathered"])



    standardized = table.copy()

    standardized["Year"] = year

    standardized["FOE"] = standardized[foe_col] if foe_col else None

    standardized["FOE_Description"] = standardized[desc_col] if desc_col else None

    standardized["FundingCluster"] = standardized[cluster_col] if cluster_col else None

    standardized["MaximumStudentContribution"] = standardized[max_student_col] if max_student_col else None

    standardized["CommonwealthContribution"] = standardized[commonwealth_col] if commonwealth_col else None



    keep_cols = [

        "Year",

        "FOE",

        "FOE_Description",

        "FundingCluster",

        "MaximumStudentContribution",

        "CommonwealthContribution",

    ]

    standardized = standardized[keep_cols].copy()

    standardized["FOE"] = standardized["FOE"].astype(str).str.strip()



    combined_rows.append(standardized)





if not combined_rows:

    raise ValueError("No year tables found to export (2019-2026).")



AnnualFundingAUS2019_2026 = pd.concat(combined_rows, ignore_index=True)

AnnualFundingAUS2019_2026 = AnnualFundingAUS2019_2026.drop_duplicates().reset_index(drop=True)



output_dir = (Path.cwd() / "data" / "clean") if (Path.cwd() / "data").exists() else (Path.cwd().parent / "data" / "clean")

output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "AnnualFundingAUS2019-2026.csv"



AnnualFundingAUS2019_2026.to_csv(output_path, index=False)



print(f"Exported: {output_path}")

print(f"Rows: {len(AnnualFundingAUS2019_2026)} | Columns: {len(AnnualFundingAUS2019_2026.columns)}")

display(AnnualFundingAUS2019_2026.head(10))


Exported: c:\Users\neddp\ECC3479-Project-JRGS\reproduction test\data\clean\AnnualFundingAUS2019-2026.csv
Rows: 3484 | Columns: 6


,Year,FOE,FOE_Description,FundingCluster,MaximumStudentContribution,CommonwealthContribution
0,2019,90701,Psychology (Not professional pathway psycholog...,Funding Cluster 3,6566,10630
1,2019,90701,Psychology (Not professional pathway psycholog...,Funding Cluster 5,6566,13073
2,2019,80100,Accounting,Funding Cluster 1,10958,2160
3,2019,80101,Accounting,Funding Cluster 1,10958,2160
4,2019,80300,Business and Management,Funding Cluster 1,10958,2160
5,2019,80301,Business Management,Funding Cluster 1,10958,2160
6,2019,80303,Human Resource Management,Funding Cluster 1,10958,2160
7,2019,80305,Personal Management Training,Funding Cluster 1,10958,2160
8,2019,80307,Organisation Management,Funding Cluster 1,10958,2160
9,2019,80309,Industrial Relations,Funding Cluster 1,10958,2160


In [3]:
# Export the same dataset to Excel

if "AnnualFundingAUS2019_2026" not in globals():

    raise ValueError("Run Cell 3 first to create AnnualFundingAUS2019_2026.")



excel_output_path = output_dir / "AnnualFundingAUS2019-2026.xlsx"

AnnualFundingAUS2019_2026.to_excel(excel_output_path, index=False)



print(f"Exported: {excel_output_path}")

print(f"Rows: {len(AnnualFundingAUS2019_2026)} | Columns: {len(AnnualFundingAUS2019_2026.columns)}")


Exported: c:\Users\neddp\ECC3479-Project-JRGS\reproduction test\data\clean\AnnualFundingAUS2019-2026.xlsx
Rows: 3484 | Columns: 6


In [4]:
# Create enrollment-aligned category keys for funding data (kept separate from existing outputs)

if "AnnualFundingAUS2019_2026" not in globals():

    raise ValueError("Run Cell 3 first to create AnnualFundingAUS2019_2026.")



category_key_to_name = {

    1: "Natural & Physical Science",

    2: "Information Technology",

    3: "Engineering & Related Tech",

    4: "Architecture & Building",

    5: "Environment & Related",

    6: "Health",

    7: "Education",

    8: "Management & Commerce",

    9: "Society & Culture",

    10: "Creative Arts",

    11: "Others",

    99: "Total",

}



broad_foe_to_category_key = {

    "01": 1,

    "02": 2,

    "03": 3,

    "04": 4,

    "05": 5,

    "06": 6,

    "07": 7,

    "08": 8,

    "09": 9,

    "10": 10,

}



def get_broad_foe(foe_value):

    text = str(foe_value).strip()

    if text.lower() in {"", "nan", "none"}:

        return None

    digits = "".join(ch for ch in text if ch.isdigit())

    if digits == "":

        return None

    return digits.zfill(6)[:2]



funding_categorised = AnnualFundingAUS2019_2026.copy()

funding_categorised["FOE_Broad"] = funding_categorised["FOE"].apply(get_broad_foe)

funding_categorised["CategoryKey"] = funding_categorised["FOE_Broad"].map(broad_foe_to_category_key).fillna(11).astype(int)

funding_categorised["Category"] = funding_categorised["CategoryKey"].map(category_key_to_name)



numeric_cols = ["MaximumStudentContribution", "CommonwealthContribution"]

for col in numeric_cols:

    funding_categorised[col] = pd.to_numeric(funding_categorised[col], errors="coerce")



AnnualFundingAUS2019_2026_with_categories = funding_categorised.copy()



category_summary = (

    funding_categorised

    .groupby(["Year", "CategoryKey", "Category"], as_index=False)

    .agg(

        Records=("FOE", "size"),

        MaximumStudentContribution=("MaximumStudentContribution", "sum"),

        CommonwealthContribution=("CommonwealthContribution", "sum"),

    )

)



total_summary = (

    funding_categorised

    .groupby(["Year"], as_index=False)

    .agg(

        Records=("FOE", "size"),

        MaximumStudentContribution=("MaximumStudentContribution", "sum"),

        CommonwealthContribution=("CommonwealthContribution", "sum"),

    )

)

total_summary["CategoryKey"] = 99

total_summary["Category"] = "Total"

total_summary = total_summary[[

    "Year",

    "CategoryKey",

    "Category",

    "Records",

    "MaximumStudentContribution",

    "CommonwealthContribution",

]]



AnnualFundingAUS2019_2026_category_summary = pd.concat([category_summary, total_summary], ignore_index=True)

AnnualFundingAUS2019_2026_category_summary = AnnualFundingAUS2019_2026_category_summary.sort_values(["Year", "CategoryKey"]).reset_index(drop=True)



categorised_csv = output_dir / "AnnualFundingAUS2019-2026_with_category_key.csv"

summary_csv = output_dir / "AnnualFundingAUS2019-2026_category_summary.csv"



AnnualFundingAUS2019_2026_with_categories.to_csv(categorised_csv, index=False)

AnnualFundingAUS2019_2026_category_summary.to_csv(summary_csv, index=False)



print(f"Exported categorised rows: {categorised_csv}")

print(f"Exported category summary: {summary_csv}")

display(AnnualFundingAUS2019_2026_category_summary.head(20))


Exported categorised rows: c:\Users\neddp\ECC3479-Project-JRGS\reproduction test\data\clean\AnnualFundingAUS2019-2026_with_category_key.csv
Exported category summary: c:\Users\neddp\ECC3479-Project-JRGS\reproduction test\data\clean\AnnualFundingAUS2019-2026_category_summary.csv


,Year,CategoryKey,Category,Records,MaximumStudentContribution,CommonwealthContribution
0,2019,1,Natural & Physical Science,37,347882,660862
1,2019,2,Information Technology,21,196539,223230
2,2019,3,Engineering & Related Tech,80,748720,1486880
3,2019,4,Architecture & Building,23,215257,244490
4,2019,5,Environment & Related,19,177821,448210
5,2019,6,Health,67,628310,1065561
6,2019,7,Education,16,105056,176976
7,2019,8,Management & Commerce,39,427362,84240
8,2019,9,Society & Culture,72,525456,614198
9,2019,10,Creative Arts,25,164150,326825


In [5]:
# Add average contribution columns per category key (and year), then re-export summary

if "AnnualFundingAUS2019_2026_category_summary" not in globals():

    raise ValueError("Run Cell 5 first to create AnnualFundingAUS2019_2026_category_summary.")



AnnualFundingAUS2019_2026_category_summary["AvgMaximumStudentContribution"] = (

    AnnualFundingAUS2019_2026_category_summary["MaximumStudentContribution"]

    / AnnualFundingAUS2019_2026_category_summary["Records"].replace(0, pd.NA)

)



AnnualFundingAUS2019_2026_category_summary["AvgCommonwealthContribution"] = (

    AnnualFundingAUS2019_2026_category_summary["CommonwealthContribution"]

    / AnnualFundingAUS2019_2026_category_summary["Records"].replace(0, pd.NA)

)



AnnualFundingAUS2019_2026_category_summary = AnnualFundingAUS2019_2026_category_summary.sort_values(["Year", "CategoryKey"]).reset_index(drop=True)



summary_csv = output_dir / "AnnualFundingAUS2019-2026_category_summary.csv"

AnnualFundingAUS2019_2026_category_summary.to_csv(summary_csv, index=False)



print(f"Updated summary export: {summary_csv}")

display(AnnualFundingAUS2019_2026_category_summary.head(20))


Updated summary export: c:\Users\neddp\ECC3479-Project-JRGS\reproduction test\data\clean\AnnualFundingAUS2019-2026_category_summary.csv


,Year,CategoryKey,Category,Records,MaximumStudentContribution,CommonwealthContribution,AvgMaximumStudentContribution,AvgCommonwealthContribution
0,2019,1,Natural & Physical Science,37,347882,660862,9402.216216,17861.135135
1,2019,2,Information Technology,21,196539,223230,9359.000000,10630.000000
2,2019,3,Engineering & Related Tech,80,748720,1486880,9359.000000,18586.000000
3,2019,4,Architecture & Building,23,215257,244490,9359.000000,10630.000000
4,2019,5,Environment & Related,19,177821,448210,9359.000000,23590.000000
5,2019,6,Health,67,628310,1065561,9377.761194,15903.895522
6,2019,7,Education,16,105056,176976,6566.000000,11061.000000
7,2019,8,Management & Commerce,39,427362,84240,10958.000000,2160.000000
8,2019,9,Society & Culture,72,525456,614198,7298.000000,8530.527778
9,2019,10,Creative Arts,25,164150,326825,6566.000000,13073.000000


In [6]:
# Quick check: what years are in the category summary?

print("In-memory summary years:", sorted(AnnualFundingAUS2019_2026_category_summary["Year"].dropna().unique().tolist()))

check_csv = output_dir / "AnnualFundingAUS2019-2026_category_summary.csv"

check_df = pd.read_csv(check_csv)

print("CSV summary years:", sorted(check_df["Year"].dropna().unique().tolist()))

print("Total rows:", len(check_df))


In-memory summary years: [2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]
CSV summary years: [2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]
Total rows: 96
